# Việc 8 — mã hoá lại keyframe bằng ViT-L-14 (Colab)

Sinh vector cho **shard của máy bạn**, lưu mỗi video một tệp `.npy` vào Google Drive.

**HYBRID, không thay thế.** Chỉ mục BTC (ViT-B/32) giữ nguyên; cái này là nhánh RRF
thứ năm, trọng số mặc định 0 cho tới khi đo được.

---
### Cần chuẩn bị

**Trên Drive**, một thư mục chứa keyframes của shard bạn giữ:
```
MyDrive/aic2026/keyframes/L23_V001/001.jpg …
MyDrive/aic2026/keyframes/L23_V002/…
```
Tên thư mục con phải đúng `video_id`. Tên tệp ảnh đệm mấy chữ số cũng được —
notebook đọc **số** trong tên, không đọc chuỗi.

**Bật GPU:** Runtime → Change runtime type → T4 GPU. Trên CPU thì mất hàng chục giờ.

### Chi phí

| | |
|---|---|
| tốc độ trên T4 | ~60–90 ảnh/giây |
| 40.000 ảnh (một shard) | ~10–15 phút |
| toàn kho 177.321 ảnh | ~40–60 phút |
| dung lượng vector | ~3 KB/ảnh, cả kho ~510 MB |

Colab miễn phí hay ngắt phiên. Notebook **chạy lại được**: video nào đã có `.npy`
thì bỏ qua, chỉ làm tiếp phần còn thiếu.


## 1. Cài đặt và kiểm GPU


In [ ]:
!pip install -q open-clip-torch

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ — đổi Runtime type')


## 2. Mount Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# SỬA HAI DÒNG NÀY cho khớp Drive của bạn
THU_MUC_ANH = Path('/content/drive/MyDrive/aic2026/keyframes')
THU_MUC_RA  = Path('/content/drive/MyDrive/aic2026/clip_l')

THU_MUC_RA.mkdir(parents=True, exist_ok=True)
video = sorted(p.name for p in THU_MUC_ANH.iterdir() if p.is_dir())
print(f'{len(video)} video: {video[:5]} …')


## 3. Nạp mô hình

`ViT-L-14/laion2b_s32b_b82k`, 768 chiều. **Đừng đổi** trừ khi cả nhóm đổi cùng lúc —
trộn hai đời vector là hỏng cả chỉ mục, và `clip_l_index.py` sẽ từ chối tệp sai số chiều.


In [ ]:
import open_clip

MO_HINH, PRETRAINED, SO_CHIEU = 'ViT-L-14', 'laion2b_s32b_b82k', 768

thiet_bi = 'cuda' if torch.cuda.is_available() else 'cpu'
model, _, preprocess = open_clip.create_model_and_transforms(
    MO_HINH, pretrained=PRETRAINED, device=thiet_bi)
model.eval()
print(MO_HINH, '|', thiet_bi)


## 4. Mã hoá

Ba điều quan trọng trong ô này:

**Thứ tự ảnh quyết định mọi thứ.** Hàng `i` của `.npy` phải ứng với keyframe thứ `i`
khi sắp theo **số** trong tên tệp. Sai thứ tự thì mọi kết quả trỏ nhầm ảnh mà không
có triệu chứng gì — đúng cái bẫy `n` với `frame_idx` của dự án.

**Ghi qua tệp tạm rồi đổi tên.** Đứt phiên giữa lúc ghi không để lại tệp cụt mà lần
sau tưởng là đã xong.

**Vector chưa chuẩn hoá.** Việc chuẩn hoá do `clip_l_index.dung_chi_muc()` làm, để
chỉ có một chỗ quyết định.


In [ ]:
import re, time
import numpy as np
from PIL import Image

LO = 64          # giảm xuống 32 nếu báo hết bộ nhớ GPU

def anh_cua(thu_muc):
    """Ảnh sắp theo SỐ trong tên tệp, không theo chuỗi."""
    ra = []
    for p in thu_muc.iterdir():
        if p.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
            continue
        k = re.search(r'(\\d+)', p.stem)
        if k:
            ra.append((int(k.group(1)), p))
    return [p for _, p in sorted(ra)]

def ma_hoa(duong_dan_anh):
    v = []
    for i in range(0, len(duong_dan_anh), LO):
        lo = duong_dan_anh[i:i + LO]
        x = torch.stack([
            preprocess(Image.open(p).convert('RGB')) for p in lo
        ]).to(thiet_bi)
        with torch.no_grad():
            v.append(model.encode_image(x).cpu().numpy().astype(np.float32))
    return np.vstack(v)


bat_dau, xong, bo_qua, tong_anh = time.time(), 0, 0, 0

for thu_tu, vid in enumerate(video, 1):
    dich = THU_MUC_RA / f'{vid}.npy'
    anh = anh_cua(THU_MUC_ANH / vid)

    if dich.exists():
        # Đã có VÀ đủ hàng thì bỏ qua. Đủ hàng mới tính là xong —
        # tệp cụt từ phiên đứt trước phải được làm lại.
        try:
            if len(np.load(dich, mmap_mode='r')) == len(anh):
                bo_qua += 1
                continue
        except Exception:
            pass

    if not anh:
        print(f'  {vid}: không có ảnh, bỏ qua')
        continue

    v = ma_hoa(anh)
    assert v.shape == (len(anh), SO_CHIEU), f'{vid}: shape {v.shape} sai'

    tam = dich.with_suffix('.tmp.npy')
    np.save(tam, v)
    tam.replace(dich)

    xong += 1; tong_anh += len(anh)
    troi = time.time() - bat_dau
    con = (len(video) - thu_tu) * troi / max(thu_tu - bo_qua, 1)
    print(f'[{thu_tu}/{len(video)}] {vid}: {len(anh)} ảnh | '
          f'{tong_anh / troi:.0f} ảnh/giây | còn ~{con / 60:.0f} phút')

print(f'\\nXong {xong} video ({tong_anh:,} ảnh), bỏ qua {bo_qua} video đã có.')


## 5. Soát trước khi tải về

Ô này kiểm số hàng và số chiều. Tệp nào lệch thì xoá đi rồi chạy lại ô 4 —
nó sẽ tự làm lại đúng tệp đó.


In [ ]:
hong, tong = [], 0
for vid in video:
    dich = THU_MUC_RA / f'{vid}.npy'
    if not dich.exists():
        hong.append((vid, 'chưa có')); continue
    v = np.load(dich, mmap_mode='r')
    can = len(anh_cua(THU_MUC_ANH / vid))
    if v.shape[1] != SO_CHIEU:
        hong.append((vid, f'{v.shape[1]} chiều, cần {SO_CHIEU}'))
    elif len(v) != can:
        hong.append((vid, f'{len(v)} hàng, cần {can}'))
    else:
        tong += len(v)

print(f'{tong:,} vector hợp lệ trên {len(video) - len(hong)}/{len(video)} video')
for vid, ly_do in hong[:20]:
    print(f'  HỎNG {vid}: {ly_do}')

dung_luong = sum((THU_MUC_RA / f'{v}.npy').stat().st_size
                 for v in video if (THU_MUC_RA / f'{v}.npy').exists())
print(f'\\nTổng {dung_luong / 1e6:.0f} MB')


## 6. Tải về máy

Vector nằm sẵn trên Drive — đồng bộ Drive về máy, hoặc tải thư mục `clip_l` xuống,
rồi chép vào:
```
D:\aic-data\derived\clip_l\
```

Gộp thư mục của **cả bốn máy** vào cùng chỗ đó (mỗi video một tệp, không đụng nhau),
rồi trên máy bất kỳ:

```powershell
python -u -m scripts.build_faiss_l --kiem     # soát trước
python -u -m scripts.build_faiss_l            # dựng chỉ mục
```

### Đo A/B
```powershell
python -u -m scripts.do_trong_so_rrf --khong-caption --nguon-mo-rong marian --chi-do-rieng
python -u -m scripts.do_trong_so_rrf --khong-caption --nguon-mo-rong marian --chi-do-rieng --clip-l
```
Mốc nền: nhánh `clip` một mình trên KIS = **0,3667**.

Nhánh `clip_l` kém hơn thì để trọng số 0 và không mất gì — đó là điểm của hybrid.

**Lưu ý cỡ mẫu:** chỉ mục phủ một phần kho thì `clip_l` vắng mặt ở các video chưa
mã hoá, và con số đo được thấp hơn thực tế. Muốn so công bằng thì lọc bộ dev xuống
những câu có trong chỉ mục.
